# CardioSentry — ConvNeXt V2 Tiny vs Nano

Trains **ConvNeXt V2-Nano** and **ConvNeXt V2-Tiny** as a controlled,
four-head CardioSentry comparison. The heads are independent sigmoids for
**STEMI, AF, LVH, and NORMAL**; AF+LVH co-occurrence is preserved.

## What is held constant

| setting | value |
|---|---|
| input | `nat768`: 768x1024 full sheet, printed ID redacted |
| split / seed | existing patient-disjoint split / 42 |
| pretraining | FCMAE self-supervised on ImageNet-1k, then ImageNet-1k fine-tune |
| optimizer | AdamW, lr 1e-4, weight decay 0.05 |
| schedule | cosine, 2 warm-up epochs |
| effective batch | 32 via gradient accumulation |
| training | up to 20 epochs, AMP, early stopping patience 5 |
| selection | best validation macro AUPRC |
| thresholds | per-head validation-F1 optimum, frozen before test |

The intended variable is only model scale: Nano is about **15.6 M** parameters
and Tiny about **28.6 M**. Both use the same timm checkpoint recipe and are fully
convolutional, so neither needs input padding or positional-embedding changes.

## Time and resilience

The notebook preflights both models on the active GPU before training. Every
epoch, status, checkpoint, metric, and prediction is written to Drive as it is
produced. Re-running the training cell resumes an interrupted model and skips a
finished one.

---
Before starting: **Runtime -> Change runtime type -> GPU**. L4 or A100 preferred.
Read the preflight VRAM table before raising the micro-batch.


## Cell 1 — unpack the training harness and install timm

The complete, self-contained harness is embedded in this cell: dataset checks,
rendering/redaction, model registry, training, checkpoint resume, evaluation,
artifact writing, comparisons, and plotting. No code-folder upload is needed.


In [ ]:
import base64, io, tarfile, os
os.makedirs("/content/code", exist_ok=True)
_PKG = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(_PKG)), mode="r:gz").extractall("/content/code")
for _f in os.listdir("/content/code"):
    if _f.endswith(".sh"):
        os.chmod(f"/content/code/{_f}", 0o755)
os.environ["CS_DIM"] = "nat768"          # fixed geometry for the controlled two-model comparison
print("unpacked:")
for f in sorted(os.listdir("/content/code")):
    print("   ", f)

!pip install -q "timm>=1.0.9"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Cell 2 — mount Drive, set caches, and extract the dataset

Add `HF_TOKEN` through Colab Secrets if available; never paste it into the
notebook. Training reads images from local `/content`, not the network-mounted
Drive. Persistent files follow the CardioSentry Drive directory map:

| folder | contents |
|---|---|
| `codes/convnextv2_tiny_nano_v1/` | place the notebook copy here |
| `datasets/` | read-only `prototype_1152.tar` |
| `experiments/convnextv2_tiny_nano_v1/` | samples and cached resized corpus |
| `runs/convnextv2_tiny_nano_v1/` | logs, metrics, predictions, weights, summaries |
| `hf_cache/` | timm/Hugging Face checkpoint cache |

The dataset is extracted to local disk; the prepared corpus is cached as one
tar archive in `experiments/` so runtime restarts do not require rebuilding it.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE = "/content/drive/MyDrive/CardioSentry"
DIM   = "nat768"
os.environ["CS_DIM"] = DIM

# Cache pretrained weights on Drive. Without this every runtime reset
# re-downloads all nine from HuggingFace - slow, and it is what triggers the
# "unauthenticated requests" rate limit over a night of reconnects.
os.environ["HF_HOME"] = f"{DRIVE}/hf_cache"

try:
    from google.colab import userdata
    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
        os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok   # older hub versions
        print("HF token loaded from Colab Secrets")
    else:
        print("HF_TOKEN secret is empty - continuing unauthenticated (works fine)")
except Exception as exc:
    print(f"no HF_TOKEN in Colab Secrets ({type(exc).__name__}) - "
          f"continuing unauthenticated (works fine)")

TAR  = f"{DRIVE}/datasets/prototype_1152.tar"
RAW  = "/content/prototype_1152"
DATA = RAW                      # cell 4 repoints this at the pre-resized copy
CODES = f"{DRIVE}/codes/convnextv2_tiny_nano_v1"
EXP  = f"{DRIVE}/experiments/convnextv2_tiny_nano_v1"
CACHE = f"{EXP}/data"                # the pre-resized corpus is cached here
RUNS = f"{DRIVE}/runs/convnextv2_tiny_nano_v1"

if not os.path.exists(f"{RAW}/index.csv"):
    !tar -xf "$TAR" -C /content/
!ls "$RAW"
!mkdir -p "$CODES" "$RUNS" "$EXP" "$CACHE"
print("\nRAW =", RAW, "\nCODES =", CODES, "\nRUNS =", RUNS, "\nEXP  =", EXP, "\nCACHE =", CACHE)

print("\nDrive space:")
!df -h /content/drive/MyDrive | tail -1
print("local disk (holds the pre-resized copy):")
!df -h /content | tail -1

## Cell 3 — see what the model will actually receive

Renders a few sheets through **the same code path training uses**, so this is
not a mock-up. Check that the redaction box covers the printed block and
nothing else — all four trace rows, twelve lead labels, both calibration pulses
and the `25mm/s 10mm/mV` footer should be untouched.

The last sheet is the **most rotated in the corpus (-3.00 deg)**, which is the
hardest case for a fixed box.

In [ ]:
%cd /content/code
!python dimensions.py | head -12

In [ ]:
import pandas as pd
from PIL import Image
from IPython.display import display, Markdown
import sys; sys.path.insert(0, "/content/code")
# `DIM` must be a geometry NAME ("nat768") for render(), never the module:
# `import dimensions as DIM` makes render() receive the module object and
# resolve() raises KeyError. Import under another name and re-derive DIM from
# DEFAULT_DIM so the call below is correct however this cell is re-run.
import dimensions as dimlib
from aspect_study import render

DIM = dimlib.DEFAULT_DIM          # the geometry NAME, e.g. "nat768"

df = pd.read_csv(f"{RAW}/index.csv")
df = df[~df.record.isin(dimlib.EXCLUDED_RECORDS)]
picks = [df[df.cls == c].iloc[0] for c in ("STEMI", "LVH", "AF", "NORMAL")]
picks.append(df.iloc[df["rotation_deg"].abs().idxmax()])      # worst rotation

os.makedirs(f"{EXP}/samples", exist_ok=True)
for r in picks:
    out = render(Image.open(f"{RAW}/{r['rel_path']}").convert("RGB"), DIM)
    p = f"{EXP}/samples/{r['cls']}_{r['record']}.png"
    out.save(p)
    display(Markdown(f"### {r['cls']}  {r['record']}  — rotation {float(r['rotation_deg']):+.2f} deg"))
    display(out)
print("saved to", f"{EXP}/samples")

## Cell 4 — prepare and cache the `nat768` corpus

The first run resizes 6,670 images to 768x1024 and bakes in the printed-ID
redaction; later runtimes restore the prepared corpus from one tar file in the
experiment folder. Training still reads from local disk. The tar carries a
geometry marker and is refused if it does not match `nat768`.


In [ ]:
%cd /content/code
!./prepare_corpus.sh "$RAW" "/content/prototype_{DIM}" "$CACHE"

# A failed shell command does NOT stop a Colab cell, so verify before switching.
_pre = f"/content/prototype_{DIM}"
if os.path.exists(os.path.join(_pre, "index.csv")):
    DATA = _pre
    print("\nDATA is now", DATA)
    !du -sh "$CACHE"/*.tar 2>/dev/null || true
else:
    DATA = RAW
    print("\nPREPARE DID NOT COMPLETE - DATA left as", DATA)
    print("training still works, just slower; read the error above")

## Cell 5 — self-test and GPU preflight (do not skip)

The self-test checks geometry, redaction, split integrity, fingerprints, and
effective-batch invariants. Preflight resolves both official timm tags and runs
a real forward/backward pass at 768x1024, reporting peak VRAM and a safe batch
suggestion before the long run begins.


In [ ]:
%cd /content/code
!python selftest.py --data "$DATA"

In [ ]:
!python preflight.py --data "$DATA"

## Cell 6 — smoke-test both models

Each command exercises 20 training and 10 evaluation batches end to end. Smoke
metrics are intentionally discarded; this cell verifies data, AMP, checkpoint,
and evaluation paths before committing to the full comparison.


In [ ]:
!python train.py --model convnextv2_nano --data "$DATA" --out /content/smoke --smoke
!python train.py --model convnextv2_tiny --data "$DATA" --out /content/smoke --smoke


## Cell 7 — train Nano and Tiny

Nano runs first. If Colab disconnects, re-run this cell: completed models are
skipped and interrupted models resume from `last.pt`. A failure in one model is
recorded without preventing the other from running.


In [ ]:
%cd /content/code
!./run_all.sh "$DATA" "$RUNS"

### Run one model at a time

Use these commented commands instead of Cell 7 when you want manual control.


In [ ]:
# !python train.py --model convnextv2_nano --data "$DATA" --out "$RUNS"
# !python train.py --model convnextv2_tiny --data "$DATA" --out "$RUNS"


## Cell 8 — progress and failure check

Safe to run while training is partial or after a resumed session.


In [ ]:
!cat "$RUNS/_SUMMARY.txt"

In [ ]:
# did anything fail overnight?
!for f in "$RUNS"/*/STATUS.txt; do echo "--- $f"; head -3 "$f"; done
!ls "$RUNS"/*/ERROR.txt 2>/dev/null && echo "^ failures above" || echo "no failures"

## Cell 9 — controlled comparison

`compare.py` verifies matching configuration fingerprints, geometry, dataset,
and row counts before ranking the models. It refuses to mix incompatible runs.


In [ ]:
%cd /content/code
!python compare.py --runs "$RUNS" --csv "$RUNS/comparison.csv"

In [ ]:
import pandas as pd
pd.set_option("display.width", 220, "display.max_columns", 60)
df = pd.read_csv(f"{RUNS}/comparison.csv")
cols = ["model", "macro_auroc", "macro_auprc", "macro_sensitivity",
        "macro_specificity", "macro_precision", "macro_f1", "macro_fpr",
        "exact_match_accuracy", "params_total_m", "size_mb", "latency_ms_b1"]
display(df[[c for c in cols if c in df.columns]]
        .sort_values("macro_auprc", ascending=False))

## Cell 10 — per-condition metrics

Prints AUROC, AUPRC, sensitivity, specificity, precision, F1, FPR, and accuracy
for each head plus the unweighted macro value. Read the minority, clinically
critical STEMI head alongside the source-confound analysis.


In [ ]:
import pandas as pd
df = pd.read_csv(f"{RUNS}/comparison.csv")
for metric in ["auroc", "auprc", "sensitivity", "specificity", "precision",
               "f1", "fpr", "accuracy"]:
    cols = [f"{l}_{metric}" for l in ["STEMI", "AF", "LVH", "NORMAL"]]
    cols = [c for c in cols if c in df.columns]
    if not cols:
        continue
    t = df.set_index("model")[cols + [f"macro_{metric}"]]
    t.columns = ["STEMI", "AF", "LVH", "NORMAL", "MACRO"][:len(t.columns)]
    print(f"\n{metric.upper()} (test)")
    print(t.sort_values("MACRO", ascending=False).round(4).to_string())

## Cell 11 — learning curves

Plots train loss, validation macro AUPRC, and validation macro AUROC per epoch.


In [ ]:
import json, glob
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(18, 4.6))
for f in sorted(glob.glob(f"{RUNS}/*/result.json")):
    r = json.load(open(f))
    h = r["history"]
    if not h or r.get("smoke"):
        continue
    ep = [x["epoch"] for x in h]
    ax[0].plot(ep, [x["train_loss"] for x in h], label=r["model"])
    ax[1].plot(ep, [x["val_macro_auprc"] for x in h], label=r["model"])
    ax[2].plot(ep, [x["val_macro_auroc"] for x in h], label=r["model"])
for a, t in zip(ax, ["train loss", "val macro AUPRC", "val macro AUROC"]):
    a.set_title(t); a.set_xlabel("epoch"); a.legend(fontsize=8); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## Cell 12 — accuracy versus deployment cost

Plots test macro AUPRC against parameter count and measured single-image
latency. Nano and Tiny receive exactly the same 768x1024 tensor, so this is a
direct scale/accuracy/cost comparison.


In [ ]:
import json, glob
import matplotlib.pyplot as plt

runs = [json.load(open(f)) for f in sorted(glob.glob(f"{RUNS}/*/result.json"))]
runs = [r for r in runs if not r.get("smoke")]

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for r in runs:
    fam = r["model_info"].get("family", "cnn")
    c = "C0" if fam == "cnn" else "C3"
    for a, x in ((ax[0], r["complexity"]["params_total_m"]),
                 (ax[1], r["inference"]["latency_ms_b1_mean"])):
        a.scatter(x, r["test"]["macro_auprc"], s=90, color=c, zorder=3)
        a.annotate(r["model"], (x, r["test"]["macro_auprc"]),
                   textcoords="offset points", xytext=(7, 4), fontsize=9)
ax[0].set_xlabel("parameters (M)"); ax[1].set_xlabel("latency, batch=1 (ms)")
for a in ax:
    a.set_ylabel("test macro AUPRC"); a.grid(alpha=.3)
ax[0].set_title("blue = CNN, red = transformer")
plt.tight_layout(); plt.show()

## Cell 13 — source-confound check (do not skip)

All STEMI records originate in one database, so a high STEMI score can partly
reflect scanner/source recognition. The printed identifier is redacted, but
residual acquisition cues remain possible. Report per-source macro AUROC and
its spread next to the headline comparison.


In [ ]:
import json, glob
for f in sorted(glob.glob(f"{RUNS}/*/result.json")):
    r = json.load(open(f))
    ps = r.get("test_per_source", {})
    if not ps:
        continue
    v = [d["macro_auroc"] for d in ps.values()]
    print(f"{r['model']:18} " +
          "  ".join(f"{s}={d['macro_auroc']:.3f}(n={d['n']})" for s, d in ps.items()) +
          f"   spread={max(v)-min(v):.3f}")

## Cell 14 — full report for one model

Change `MODEL` to inspect the complete plain-text validation/test report.


In [ ]:
import glob
MODEL = "convnextv2_nano"          # or "convnextv2_tiny"
hits = glob.glob(f"{RUNS}/{MODEL}@*/report.txt")
print(open(hits[0]).read() if hits else f"no finished run for {MODEL}")


## Where everything lives

```text
/content/drive/MyDrive/CardioSentry/
├── codes/convnextv2_tiny_nano_v1/
│   └── CardioSentry_ConvNeXtV2_Tiny_Nano.ipynb
├── experiments/convnextv2_tiny_nano_v1/
│   ├── samples/                         rendered QA sheets
│   └── data/                            cached nat768 corpus tar
├── hf_cache/                            pretrained checkpoint cache
└── runs/convnextv2_tiny_nano_v1/
    ├── _SUMMARY.txt                     both models at a glance
    ├── _batch_log.txt
    ├── comparison.txt / comparison.csv
    └── <model>@nat768/
        ├── STATUS.txt                   RUNNING / DONE / FAILED
        ├── train_log.txt                flushed continuously
        ├── history.csv                  one row per completed epoch
        ├── metrics.csv                  validation/test, label + macro rows
        ├── report.txt                   human-readable full report
        ├── result.json                  machine-readable complete result
        ├── config_snapshot.json         settings + fingerprints
        ├── weights/best.pt              best weights + frozen thresholds
        └── predictions/
            ├── val_predictions.csv
            └── test_predictions.csv
```

`last.pt` exists only while a run is resumable and is deleted after success;
`best.pt` is retained. Final metrics include per-label and macro **AUROC, AUPRC,
sensitivity, specificity, precision, NPV, F1, FPR, FNR, accuracy, balanced
accuracy, thresholds, support, and TP/FP/FN/TN**; micro precision/sensitivity/
specificity/F1; exact-match and Hamming accuracy; per-source breakdowns; model
size/parameter counts; batch-1 latency (mean, p50, p95); and batched throughput.

## Interpretation limits

1. This is one seed on the prototype subset; it does not estimate run-to-run
   variance or confidence intervals.
2. Validation chooses the checkpoint and per-head thresholds; test is evaluated
   once with those thresholds frozen.
3. All STEMI records come from one source. The source-spread cell is mandatory,
   and cross-source validation remains outstanding.
4. Thresholds, calibration, and absolute scores from the one-augmentation
   prototype should not be presented as final deployment estimates.
5. The older `v1`/`v1_768` STEMI numbers retained a printed identifier that was
   perfectly source-linked; do not compare their STEMI scores directly here.
